## Installing necessary libraries

Cohere provides free trial keys to use their LLMs. So generate one trial key from dashboard.cohere.com

In [1]:
!pip install langchain-openai langchain pdfminer.six chromadb langchain-community langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_openai-1.6.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.6.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached openai-3.13.0-py3-none-any.whl.metadata (41 kB)
  Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pybase64-1.5.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-non

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openai-agents

langchain-openai: Enables integration of OpenAI's language and embedding models with LangChain for advanced text generation and processing workflows.

langchain: Provides a modular framework for building language model-powered applications, such as chatbots, question-answering systems, and conversational agents.

pdfminer.six: Facilitates text extraction from PDF files, making it useful for document analysis and preprocessing tasks.

chromadb: A vector database library designed for efficient storage and retrieval of embeddings, ideal for tasks like semantic search and recommendation systems.

## Importing libraries

In [2]:
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    pass  # not running in Colab -- assume OPENAI_API_KEY is already set in the environment

from langchain_core.prompts import PromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pdfminer.high_level import extract_text as extract_text_pdf_miner
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

C:\Users\Avado\AppData\Local\Temp\ipykernel_35188\2442543177.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


## VectorDB setup

In [3]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/chroma_db"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

We are processing 2 research papers on transformers and yolo. You can use the PDFs

In [4]:
# Loop through a list of PDF files to process
for pdf_name in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,
        persist_directory=persist_directory,
        embedding=embedding
    )

In [5]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

C:\Users\Avado\AppData\Local\Temp\ipykernel_35188\1816146847.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [6]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

[(Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results. YOLO running on sample artwork and natural images from the internet. It is mostly accurate a

## Chain Setup

In [7]:
# Initialize an LLM instance using OpenAI's "gpt-4o-mini" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [8]:
# Define a prompt template for generating answers based on a given context and question
prompt_str = """Given a chat history and the latest user question which might reference context in the chat history,
formulate a standalone question which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

# Use a prompt that includes a MessagesPlaceholder variable under the name "chat_history".
# This allows us to pass in a list of Messages to the prompt using the "chat_history" input key,
# and these messages will be inserted after the system message and before the human message containing the
# latest question.

prompt_history_aware = ChatPromptTemplate.from_messages([
    ("system", prompt_str),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# create_history_aware_retriever constructs a chain that accepts keys input and chat_history as input, and has the same output schema as a retriever
history_aware_retriever = create_history_aware_retriever(
    llm, vectordb.as_retriever(), prompt_history_aware
)

Now its time to build our final rag_chain with create_retrieval_chain. This chain applies the history_aware_retriever and question_answer_chain created with create_stuff_documents_chain in sequence, retaining intermediate outputs such as the retrieved context for convenience. It has input keys input and chat_history, and includes input, chat_history, context, and answer in its output.



In [9]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question. If you don't know the answer,
say that you don't know. Use three sentences maximum and keep theanswer concise.

{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# create_retrieval_chain combines history aware retriever and the qa chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [10]:
# Load a empty chat_history list
chat_history = []

# Invoking the chain
question = "What is YOLO?"
response = rag_chain.invoke({"input": question, "chat_history": chat_history})

# Appeding question and response answers
chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response["answer"]),
    ]
)

In [11]:
#check chat_history
chat_history

[HumanMessage(content='What is YOLO?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='YOLO, which stands for "You Only Look Once," is a unified model for real-time object detection that frames the task as a regression problem. It uses a single convolutional neural network to predict multiple bounding boxes and class probabilities directly from full images in one evaluation. YOLO is known for its speed and accuracy, processing images at up to 155 frames per second while achieving high mean average precision.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [12]:
second_question = "What are the tasks YOLO can be used in?"
response = rag_chain.invoke({"input": second_question, "chat_history": chat_history})

print(response["answer"])

YOLO can be used in various computer vision applications, including real-time object detection in video streams, autonomous driving, and assistive devices that provide scene information. Its fast and accurate detection capabilities make it suitable for tasks that require quick responses, such as tracking moving objects. Additionally, YOLO can generalize well to different domains, making it applicable in areas like artwork recognition and surveillance.


To automate the inserting and updating of chat history. And have a session id that can be unique for a user

In [13]:
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

C:\Users\Avado\AppData\Local\Temp\claude\C--Users-Avado-Desktop-LearningGenAI\37e7fada-b4a3-4f8e-b68e-c30c95841747\scratchpad\notebook_run\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
conversational_rag_chain.invoke(
    {"input": "What is Transformers?"},
    config={
        "configurable": {"session_id": "User_1"}
    },
)["answer"]

'Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language translation, that rely entirely on self-attention mechanisms instead of recurrent or convolutional layers. This architecture allows for greater parallelization and efficiency in training, achieving state-of-the-art results in various natural language processing tasks. The Transformer model consists of an encoder-decoder structure, where the encoder processes input sequences and the decoder generates output sequences.'

In [15]:
# This will output input, chat_history, contexts and answer
conversational_rag_chain.invoke(
    {"input": "Transformers vs YOLO"},
    config={
        "configurable": {"session_id": "User_1"}
    },
)

{'input': 'Transformers vs YOLO',
 'chat_history': [HumanMessage(content='What is Transformers?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language translation, that rely entirely on self-attention mechanisms instead of recurrent or convolutional layers. This architecture allows for greater parallelization and efficiency in training, achieving state-of-the-art results in various natural language processing tasks. The Transformer model consists of an encoder-decoder structure, where the encoder processes input sequences and the decoder generates output sequences.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In r

In [16]:
# Check the chat_history of the store dict
for message in store["User_1"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

User: What is Transformers?

AI: Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language translation, that rely entirely on self-attention mechanisms instead of recurrent or convolutional layers. This architecture allows for greater parallelization and efficiency in training, achieving state-of-the-art results in various natural language processing tasks. The Transformer model consists of an encoder-decoder structure, where the encoder processes input sequences and the decoder generates output sequences.

User: Transformers vs YOLO

AI: Transformers and YOLO serve different purposes in the field of machine learning. Transformers are primarily used for natural language processing tasks, leveraging self-attention mechanisms to understand context in sequences, while YOLO (You Only Look Once) is a real-time object detection system that frames detection as a regression problem to predict bounding boxes and class probabilities directl